In [ ]:

# ================================================================
# ARSO-OPTIMIZED 5D LOGISTIC CHAOTIC MAP FOR IMAGE ENCRYPTION
# ================================================================
# Workflow:
# 1. Load image dataset
# 2. Split into 70% training, 15% validation, 15% testing
# 3. Image preprocessing
# 4. Generate coupled 5D logistic chaotic sequences
# 5. ARSO-inspired parameter optimization
# 6. Plaintext-dependent initial-state generation
# 7. Confusion and diffusion
# 8. Image encryption and decryption
# 9. Security evaluation
# ================================================================

import csv
import hashlib
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split


# ================================================================
# 1. CONFIGURATION
# ================================================================

DATASET_DIR = Path("./image_dataset")
OUTPUT_DIR = Path("./ARSO_5D_Results")

IMAGE_SIZE = (256, 256)
SPLIT_SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

MASTER_KEY = "CHANGE_TO_YOUR_PRIVATE_SECRET_KEY"

POPULATION_SIZE = 8
ARSO_ITERATIONS = 10
CHAOTIC_BURN_IN = 500

OPTIMIZATION_IMAGES = 4
FITNESS_SEQUENCE_LENGTH = 2048

random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
}


# ================================================================
# 2. DATASET LOADING
# ================================================================

def load_dataset(folder):

    image_files = sorted([
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    if len(image_files) < 4:
        raise ValueError(
            "At least four images are required for "
            "training, validation and testing."
        )

    return image_files


def split_dataset(image_files):

    train_files, temporary_files = train_test_split(
        image_files,
        test_size=0.30,
        random_state=SPLIT_SEED,
        shuffle=True
    )

    val_files, test_files = train_test_split(
        temporary_files,
        test_size=0.50,
        random_state=SPLIT_SEED,
        shuffle=True
    )

    return train_files, val_files, test_files


# ================================================================
# 3. IMAGE PREPROCESSING
# ================================================================

def preprocess_image(image_path):

    image = Image.open(image_path)

    if image.mode not in ["L", "RGB"]:
        image = image.convert("RGB")

    image = image.resize(
        IMAGE_SIZE,
        Image.Resampling.LANCZOS
    )

    return np.asarray(image, dtype=np.uint8)


def save_image(image, path):

    Path(path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    Image.fromarray(
        np.asarray(image, dtype=np.uint8)
    ).save(path)


# ================================================================
# 4. PLAINTEXT-DEPENDENT INITIAL STATE GENERATION
# ================================================================

def generate_initial_state(image):

    image_bytes = np.asarray(
        image, dtype=np.uint8
    ).tobytes()

    digest = hashlib.sha256(
        MASTER_KEY.encode("utf-8") + image_bytes
    ).digest()

    values = np.frombuffer(
        digest[:20], dtype=np.uint32
    )

    state = (
        values.astype(np.float64) + 0.5
    ) / (2**32)

    return np.clip(
        state, 1e-10, 1.0 - 1e-10
    )


# ================================================================
# 5. HIGHER-ORDER COUPLED 5D LOGISTIC CHAOTIC MAP
# ================================================================

def logistic_map_step(state, params):

    r1, r2, r3, r4, r5, coupling = params

    x1, x2, x3, x4, x5 = state

    new_state = np.zeros(5, dtype=np.float64)

    new_state[0] = (
        r1 * x1 * (1 - x1)
        + coupling * x2
        + coupling * np.sin(2 * np.pi * x5)
    ) % 1.0

    new_state[1] = (
        r2 * x2 * (1 - x2)
        + coupling * x3
        + coupling * np.sin(2 * np.pi * x1)
    ) % 1.0

    new_state[2] = (
        r3 * x3 * (1 - x3)
        + coupling * x4
        + coupling * np.sin(2 * np.pi * x2)
    ) % 1.0

    new_state[3] = (
        r4 * x4 * (1 - x4)
        + coupling * x5
        + coupling * np.sin(2 * np.pi * x3)
    ) % 1.0

    new_state[4] = (
        r5 * x5 * (1 - x5)
        + coupling * x1
        + coupling * np.sin(2 * np.pi * x4)
    ) % 1.0

    return np.clip(
        new_state, 1e-12, 1.0 - 1e-12
    )


def generate_5d_chaotic_sequence(
    params, initial_state, length
):

    state = np.asarray(
        initial_state, dtype=np.float64
    ).copy()

    sequence = np.zeros(
        (length + CHAOTIC_BURN_IN, 5),
        dtype=np.float64
    )

    for i in range(len(sequence)):

        state = logistic_map_step(
            state, params
        )

        sequence[i] = state

    sequence = sequence[CHAOTIC_BURN_IN:]

    return sequence


def chaotic_sequence_to_bytes(sequence):

    return np.floor(
        sequence * (2**32)
    ).astype(np.uint64).astype(np.uint8)


# ================================================================
# 6. STATISTICAL RANDOMNESS AND ENTROPY
# ================================================================

def calculate_entropy(data):

    data = np.asarray(
        data, dtype=np.uint8
    ).ravel()

    histogram = np.bincount(
        data, minlength=256
    )

    probabilities = histogram / len(data)

    probabilities = probabilities[
        probabilities > 0
    ]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return float(entropy)


def calculate_correlation(x, y):

    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()

    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0

    value = np.corrcoef(x, y)[0, 1]

    return float(value) if np.isfinite(value) else 0.0


def estimate_lyapunov(params, initial_state):

    epsilon = 1e-8

    x = np.asarray(
        initial_state, dtype=np.float64
    ).copy()

    y = x.copy()
    y[0] += epsilon

    total = 0.0
    valid = 0

    for _ in range(100):

        x = logistic_map_step(x, params)
        y = logistic_map_step(y, params)

        difference = y - x

        distance = np.linalg.norm(difference)

        if distance < 1e-15:
            continue

        total += np.log(distance / epsilon)

        valid += 1

        difference = (
            difference / distance
        ) * epsilon

        y = np.clip(
            x + difference,
            1e-12,
            1.0 - 1e-12
        )

    return float(total / max(valid, 1))


# ================================================================
# 7. MULTI-OBJECTIVE FITNESS FUNCTION
# ================================================================

def fitness_function(params, training_images):

    entropy_values = []
    correlation_values = []

    initial_state = np.array([
        0.123, 0.317, 0.529, 0.731, 0.947
    ])

    for image in training_images:

        image_state = generate_initial_state(image)

        sequence = generate_5d_chaotic_sequence(
            params,
            image_state,
            FITNESS_SEQUENCE_LENGTH
        )

        byte_sequence = chaotic_sequence_to_bytes(
            sequence
        )

        entropy_values.append(
            calculate_entropy(byte_sequence)
        )

        flat = byte_sequence.ravel()

        correlation_values.append(
            abs(calculate_correlation(
                flat[:-1], flat[1:]
            ))
        )

    mean_entropy = np.mean(entropy_values)

    mean_correlation = np.mean(
        correlation_values
    )

    lyapunov = estimate_lyapunov(
        params, initial_state
    )

    # Normalize the three objectives.
    entropy_score = mean_entropy / 8.0

    lyapunov_score = np.tanh(
        max(lyapunov, 0.0)
    )

    randomness_score = (
        1.0 - min(mean_correlation, 1.0)
    )

    # Higher fitness is preferred.
    fitness = (
        0.40 * entropy_score
        + 0.35 * lyapunov_score
        + 0.25 * randomness_score
    )

    return float(fitness)


# ================================================================
# 8. ARSO-INSPIRED OPTIMIZATION
# ================================================================

LOWER_BOUNDS = np.array(
    [3.60, 3.60, 3.60, 3.60, 3.60, 0.001]
)

UPPER_BOUNDS = np.array(
    [3.99, 3.99, 3.99, 3.99, 3.99, 0.035]
)


def initialize_population():

    population = []

    for _ in range(POPULATION_SIZE):

        candidate = np.random.uniform(
            LOWER_BOUNDS,
            UPPER_BOUNDS
        )

        population.append(candidate)

    return np.array(population)


def arso_optimization(training_images):

    population = initialize_population()

    fitness_values = np.array([
        fitness_function(p, training_images)
        for p in population
    ])

    best_index = np.argmax(fitness_values)

    best_solution = population[
        best_index
    ].copy()

    best_fitness = fitness_values[
        best_index
    ]

    history = []

    print("\nARSO OPTIMIZATION STARTED")

    for iteration in range(ARSO_ITERATIONS):

        progress = (
            iteration + 1
        ) / ARSO_ITERATIONS

        exploration = 1.0 - progress

        new_population = []

        for i in range(POPULATION_SIZE):

            current = population[i]

            peer = population[
                np.random.randint(POPULATION_SIZE)
            ]

            random_vector = np.random.uniform(
                -1, 1, size=6
            )

            step = (
                0.18 * exploration + 0.015
            ) * random_vector

            attraction = (
                np.random.rand(6)
                * (best_solution - current)
            )

            peer_effect = (
                0.10 * exploration
                * (peer - current)
            )

            candidate = (
                current
                + attraction * (0.35 + exploration)
                + peer_effect
                + step
            )

            candidate = np.clip(
                candidate,
                LOWER_BOUNDS,
                UPPER_BOUNDS
            )

            new_population.append(candidate)

        new_population = np.array(
            new_population
        )

        new_fitness = np.array([
            fitness_function(p, training_images)
            for p in new_population
        ])

        # Greedy selection.
        for i in range(POPULATION_SIZE):

            if new_fitness[i] >= fitness_values[i]:

                population[i] = new_population[i]

                fitness_values[i] = new_fitness[i]

        current_best = np.argmax(
            fitness_values
        )

        if fitness_values[current_best] > best_fitness:

            best_fitness = fitness_values[
                current_best
            ]

            best_solution = population[
                current_best
            ].copy()

        history.append({
            "Iteration": iteration + 1,
            "Best_Fitness": best_fitness
        })

        print(
            f"Iteration {iteration + 1}/{ARSO_ITERATIONS} "
            f"Fitness: {best_fitness:.6f}"
        )

    return best_solution, history


# ================================================================
# 9. VALIDATION FITNESS
# ================================================================

def validate_parameters(params, validation_images):

    scores = []

    for image in validation_images:

        score = fitness_function(
            params, [image]
        )

        scores.append(score)

    return float(np.mean(scores))


# ================================================================
# 10. PLAINTEXT-DEPENDENT CHAOTIC KEYSTREAM
# ================================================================

def generate_key_stream(image, params):

    initial_state = generate_initial_state(
        image
    )

    sequence = generate_5d_chaotic_sequence(
        params,
        initial_state,
        np.asarray(image).size
    )

    return (
        chaotic_sequence_to_bytes(sequence),
        initial_state
    )


# ================================================================
# 11. CONFUSION AND DIFFUSION ENCRYPTION
# ================================================================

def encrypt_image(plaintext, params):

    original_shape = plaintext.shape

    flat = np.asarray(
        plaintext, dtype=np.uint8
    ).ravel()

    streams, initial_state = generate_key_stream(
        plaintext, params
    )

    # CONFUSION: chaotic permutation.
    permutation = np.argsort(
        streams[:, 0], kind="stable"
    )

    permuted = flat[permutation]

    # DIFFUSION: chaotic XOR and chained diffusion.
    mixed = np.bitwise_xor(
        permuted, streams[:, 1]
    )

    cipher = np.empty_like(mixed)

    previous = int(streams[0, 2])

    for i in range(len(mixed)):

        cipher[i] = (
            mixed[i]
            ^ previous
            ^ streams[i, 3]
        )

        previous = int(cipher[i])

    encrypted = cipher.reshape(
        original_shape
    )

    return encrypted, initial_state


# ================================================================
# 12. IMAGE DECRYPTION
# ================================================================

def decrypt_image(ciphertext, initial_state, params):

    original_shape = ciphertext.shape

    cipher = np.asarray(
        ciphertext, dtype=np.uint8
    ).ravel()

    sequence = generate_5d_chaotic_sequence(
        params,
        initial_state,
        len(cipher)
    )

    streams = chaotic_sequence_to_bytes(
        sequence
    )

    # Reverse diffusion.
    mixed = np.empty_like(cipher)

    previous = int(streams[0, 2])

    for i in range(len(cipher)):

        mixed[i] = (
            cipher[i]
            ^ previous
            ^ streams[i, 3]
        )

        previous = int(cipher[i])

    # Reverse XOR.
    permuted = np.bitwise_xor(
        mixed, streams[:, 1]
    )

    # Reverse permutation.
    permutation = np.argsort(
        streams[:, 0], kind="stable"
    )

    restored = np.empty_like(permuted)

    restored[permutation] = permuted

    return restored.reshape(original_shape)


# ================================================================
# 13. SECURITY EVALUATION METRICS
# ================================================================

def calculate_psnr(original, encrypted):

    original = np.asarray(
        original, dtype=np.float64
    )

    encrypted = np.asarray(
        encrypted, dtype=np.float64
    )

    mse = np.mean(
        (original - encrypted) ** 2
    )

    if mse == 0:
        return float("inf")

    return float(
        10 * np.log10((255 ** 2) / mse)
    )


def calculate_npcr_uaci(cipher1, cipher2):

    a = np.asarray(
        cipher1, dtype=np.int16
    )

    b = np.asarray(
        cipher2, dtype=np.int16
    )

    if a.shape != b.shape:
        raise ValueError(
            "Cipher images must have equal dimensions."
        )

    npcr = (
        np.sum(a != b) / a.size
    ) * 100

    uaci = (
        np.mean(np.abs(a - b)) / 255
    ) * 100

    return float(npcr), float(uaci)


def calculate_adjacent_correlation(image):

    if image.ndim == 3:

        gray = np.asarray(
            Image.fromarray(image).convert("L"),
            dtype=np.float64
        )

    else:

        gray = image.astype(np.float64)

    horizontal = calculate_correlation(
        gray[:, :-1], gray[:, 1:]
    )

    vertical = calculate_correlation(
        gray[:-1, :], gray[1:, :]
    )

    diagonal = calculate_correlation(
        gray[:-1, :-1], gray[1:, 1:]
    )

    return horizontal, vertical, diagonal


def change_one_pixel(image):

    changed = image.copy()

    if changed.ndim == 2:

        changed[0, 0] = (
            int(changed[0, 0]) + 1
        ) % 256

    else:

        changed[0, 0, 0] = (
            int(changed[0, 0, 0]) + 1
        ) % 256

    return changed


# ================================================================
# 14. COMPLETE IMAGE ENCRYPTION EXPERIMENT
# ================================================================

def evaluate_image(image, params, filename):

    start_time = time.perf_counter()

    encrypted, initial_state = encrypt_image(
        image, params
    )

    encryption_time = (
        time.perf_counter() - start_time
    )

    decrypted = decrypt_image(
        encrypted,
        initial_state,
        params
    )

    recovery = np.array_equal(
        image, decrypted
    )

    # Differential attack evaluation.
    changed_image = change_one_pixel(image)

    changed_cipher, _ = encrypt_image(
        changed_image, params
    )

    npcr, uaci = calculate_npcr_uaci(
        encrypted, changed_cipher
    )

    # Ciphertext statistics.
    entropy = calculate_entropy(encrypted)

    h_corr, v_corr, d_corr = (
        calculate_adjacent_correlation(
            encrypted
        )
    )

    psnr = calculate_psnr(
        image, encrypted
    )

    return {
        "Image": filename,
        "Entropy": entropy,
        "NPCR": npcr,
        "UACI": uaci,
        "PSNR": psnr,
        "Horizontal_Correlation": h_corr,
        "Vertical_Correlation": v_corr,
        "Diagonal_Correlation": d_corr,
        "Encryption_Time": encryption_time,
        "Exact_Decryption": recovery
    }, encrypted, decrypted


# ================================================================
# 15. MAIN EXECUTION
# ================================================================

def main():

    OUTPUT_DIR.mkdir(
        parents=True, exist_ok=True
    )

    # Load dataset.
    image_files = load_dataset(
        DATASET_DIR
    )

    # Dataset split.
    train_files, val_files, test_files = (
        split_dataset(image_files)
    )

    print("\n========== DATASET SPLIT ==========")

    print("Total images:", len(image_files))

    print(
        "Training images:",
        len(train_files),
        f"({len(train_files)/len(image_files)*100:.2f}%)"
    )

    print(
        "Validation images:",
        len(val_files),
        f"({len(val_files)/len(image_files)*100:.2f}%)"
    )

    print(
        "Testing images:",
        len(test_files),
        f"({len(test_files)/len(image_files)*100:.2f}%)"
    )

    # Save split information.
    split_rows = []

    for name, files in [
        ("Training", train_files),
        ("Validation", val_files),
        ("Testing", test_files)
    ]:

        for file in files:

            split_rows.append({
                "Image": file.name,
                "Split": name
            })

    pd.DataFrame(split_rows).to_csv(
        OUTPUT_DIR / "Dataset_Split.csv",
        index=False
    )

    # Preprocess training images.
    train_images = [
        preprocess_image(p)
        for p in train_files[:OPTIMIZATION_IMAGES]
    ]

    val_images = [
        preprocess_image(p)
        for p in val_files
    ]

    # ARSO optimization.
    print("\n========== ARSO OPTIMIZATION ==========")

    best_params, history = arso_optimization(
        train_images
    )

    print("\nOptimized 5D parameters:")

    print("r1, r2, r3, r4, r5, coupling")

    print(best_params)

    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "ARSO_Optimization.csv",
        index=False
    )

    # Validation.
    print("\n========== VALIDATION ==========")

    validation_fitness = validate_parameters(
        best_params,
        val_images
    )

    print(
        "Validation fitness:",
        validation_fitness
    )

    # Save optimized parameters.
    pd.DataFrame(
        [best_params],
        columns=[
            "r1", "r2", "r3", "r4", "r5",
            "Coupling"
        ]
    ).to_csv(
        OUTPUT_DIR / "Optimized_Parameters.csv",
        index=False
    )

    # Final testing.
    print("\n========== TESTING ==========")

    results = []

    for index, image_path in enumerate(
        test_files, start=1
    ):

        image = preprocess_image(
            image_path
        )

        metrics, encrypted, decrypted = (
            evaluate_image(
                image,
                best_params,
                image_path.name
            )
        )

        results.append(metrics)

        save_image(
            encrypted,
            OUTPUT_DIR / "Encrypted" /
            f"{image_path.stem}_encrypted.png"
        )

        save_image(
            decrypted,
            OUTPUT_DIR / "Decrypted" /
            f"{image_path.stem}_decrypted.png"
        )

        print(
            f"\nTest image {index}: {image_path.name}"
        )

        for key, value in metrics.items():

            print(f"{key}: {value}")

    # Save final test metrics.
    results_df = pd.DataFrame(results)

    results_df.to_csv(
        OUTPUT_DIR / "Test_Results.csv",
        index=False
    )

    # Average metrics.
    numeric_columns = [
        "Entropy",
        "NPCR",
        "UACI",
        "PSNR",
        "Horizontal_Correlation",
        "Vertical_Correlation",
        "Diagonal_Correlation",
        "Encryption_Time"
    ]

    averages = results_df[
        numeric_columns
    ].mean()

    averages.to_frame(
        "Average"
    ).to_csv(
        OUTPUT_DIR / "Average_Results.csv"
    )

    print("\n========== FINAL AVERAGE RESULTS ==========")

    print(averages)

    print("\nAll experiments completed.")

    print(
        "Output directory:",
        OUTPUT_DIR.resolve()
    )


# ================================================================
# 16. START PROGRAM
# ================================================================

if __name__ == "__main__":

    main()
